In [ ]:
# Basic libraries for data handling and checking files
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# Basic plotting libraries
import matplotlib.pyplot as plt

In [ ]:
# Ignore warning messages for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
print("Basic libraries loaded successfully")

In [ ]:
# Check the notebook is running from the folder that holds the MIMIC-IV files
# (MIMIC-IV data are NOT included in this repository - see README for access)
current_folder = Path.cwd()
print("Working folder set.")


In [ ]:
required_files = [
    "patients.csv.gz",
    "admissions.csv.gz",
    "diagnoses_icd.csv.gz",
    "d_icd_diagnoses.csv.gz",
    "d_labitems.csv.gz",
    "labevents.csv.gz"
]

print("Checking only required MIMIC-IV files:\n")

for file in required_files:
    if Path(file).exists():
        print("FOUND:", file)
    else:
        print("MISSING:", file)

In [ ]:
# Load the small MIMIC-IV hospital files
# NOT loading labevents yet because it is very large.

patients = pd.read_csv("patients.csv.gz")
admissions = pd.read_csv("admissions.csv.gz")
diagnoses = pd.read_csv("diagnoses_icd.csv.gz")
d_icd = pd.read_csv("d_icd_diagnoses.csv.gz")
d_labs = pd.read_csv("d_labitems.csv.gz")

print("Small files loaded successfully")

print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("diagnoses:", diagnoses.shape)
print("d_icd:", d_icd.shape)
print("d_labs:", d_labs.shape)

In [ ]:
# Check the column names in each small table

print("PATIENTS columns:")
print(patients.columns.tolist())

print("\nADMISSIONS columns:")
print(admissions.columns.tolist())

print("\nDIAGNOSES columns:")
print(diagnoses.columns.tolist())

print("\nICD DICTIONARY columns:")
print(d_icd.columns.tolist())

print("\nLAB DICTIONARY columns:")
print(d_labs.columns.tolist())

In [ ]:
# Create heart failure cohort using ICD diagnosis codes

diagnoses["icd_code"] = diagnoses["icd_code"].astype(str).str.upper()

heart_failure_diagnoses = diagnoses[
    (
        (diagnoses["icd_version"] == 9) &
        (diagnoses["icd_code"].str.startswith("428"))
    )
    |
    (
        (diagnoses["icd_version"] == 10) &
        (diagnoses["icd_code"].str.startswith("I50"))
    )
].copy()

print("Heart failure diagnosis rows:", heart_failure_diagnoses.shape[0])
print("Unique heart failure patients:", heart_failure_diagnoses["subject_id"].nunique())
print("Unique heart failure admissions:", heart_failure_diagnoses["hadm_id"].nunique())

heart_failure_diagnoses.head()

In [ ]:
# Add full diagnosis names to the heart failure diagnosis table

heart_failure_diagnoses_named = heart_failure_diagnoses.merge(
    d_icd,
    on=["icd_code", "icd_version"],
    how="left"
)

print("Shape after adding diagnosis names:", heart_failure_diagnoses_named.shape)

heart_failure_diagnoses_named[
    ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version", "long_title"]
].head(10)

In [ ]:
# Merge heart failure diagnoses with admissions and patient information

hf_base = heart_failure_diagnoses_named.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="left"
)

hf_base = hf_base.merge(
    patients,
    on="subject_id",
    how="left"
)

print("Heart failure base dataset shape:", hf_base.shape)

hf_base[
    [
        "subject_id", "hadm_id", "icd_code", "long_title",
        "admittime", "dischtime", "deathtime",
        "admission_type", "race", "hospital_expire_flag",
        "gender", "anchor_age"
    ]
].head(10)

In [ ]:
# Create one row per heart failure hospital admission

hf_admissions = hf_base.drop_duplicates(
    subset=["subject_id", "hadm_id"]
).copy()

print("Original HF diagnosis-level rows:", hf_base.shape[0])
print("Unique HF admission-level rows:", hf_admissions.shape[0])
print("Unique HF patients:", hf_admissions["subject_id"].nunique())

hf_admissions[
    [
        "subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
        "admission_type", "race", "hospital_expire_flag",
        "gender", "anchor_age"
    ]
].head(10)

In [ ]:
# Create basic outcome and time variables

hf_admissions["admittime"] = pd.to_datetime(hf_admissions["admittime"])
hf_admissions["dischtime"] = pd.to_datetime(hf_admissions["dischtime"])
hf_admissions["deathtime"] = pd.to_datetime(hf_admissions["deathtime"])

hf_admissions["length_of_stay_days"] = (
    hf_admissions["dischtime"] - hf_admissions["admittime"]
).dt.total_seconds() / (60 * 60 * 24)

# MIMIC-IV anchor_age is age in anchor_year, NOT necessarily at admission.
# The 90+ ages are de-identified, so age_at_admission is not an exact age
# for the oldest patients. Relative year differences are preserved per patient.
hf_admissions["age_at_admission"] = (
    pd.to_numeric(hf_admissions["anchor_age"], errors="coerce")
    + hf_admissions["admittime"].dt.year
    - pd.to_numeric(hf_admissions["anchor_year"], errors="coerce")
)
if hf_admissions["age_at_admission"].isna().any():
    raise ValueError("Cannot calculate age at admission for all admissions.")

hf_admissions["hospital_mortality"] = hf_admissions["hospital_expire_flag"]

print("Hospital mortality counts:")
print(hf_admissions["hospital_mortality"].value_counts())

print("\nLength of stay summary:")
print(hf_admissions["length_of_stay_days"].describe())

hf_admissions[
    [
        "subject_id", "hadm_id", "admittime", "dischtime",
        "length_of_stay_days", "hospital_mortality",
        "gender", "anchor_age"
    ]
].head(10)

In [ ]:
# Search the lab dictionary for clinically relevant heart failure biomarkers

search_terms = [
    "creatinine",
    "urea",
    "bun",
    "sodium",
    "potassium",
    "chloride",
    "bicarbonate",
    "glucose",
    "hemoglobin",
    "haemoglobin",
    "hematocrit",
    "white blood",
    "wbc",
    "platelet",
    "albumin",
    "troponin",
    "bnp",
    "nt-probnp"
]

for term in search_terms:
    print("\n---", term.upper(), "---")
    display(
        d_labs[
            d_labs["label"].str.contains(term, case=False, na=False)
        ][["itemid", "label", "fluid", "category"]]
    )

In [ ]:
# Select the main blood-based biomarkers for the heart failure project

selected_biomarkers = {
    "creatinine": [50912],
    "urea_nitrogen_bun": [51006],
    "sodium": [50983],
    "potassium": [50971],
    "chloride": [50902],
    "bicarbonate": [50882],
    "glucose": [50931],
    "hemoglobin": [51222],
    "wbc_count": [51300],
    "platelet_count": [51265],
    "albumin": [50862],
    "troponin_i": [51002],
    "troponin_t": [51003],
    "ntprobnp": [50963]
}

selected_itemids = []

for biomarker, itemids in selected_biomarkers.items():
    selected_itemids.extend(itemids)

selected_labs = d_labs[d_labs["itemid"].isin(selected_itemids)].copy()

print("Number of selected lab itemids:", len(selected_itemids))

selected_labs.sort_values("itemid")

In [ ]:
# Check the structure of the big labevents file without loading the full file

labevents_sample = pd.read_csv("labevents.csv.gz", nrows=5)

print("Labevents columns:")
print(labevents_sample.columns.tolist())

labevents_sample.head()

In [ ]:
# Extract selected biomarker rows from the big labevents file safely using chunks

hf_hadm_ids = set(hf_admissions["hadm_id"].dropna().unique())
selected_itemids_set = set(selected_itemids)

use_columns = [
    "subject_id",
    "hadm_id",
    "itemid",
    "charttime",
    "valuenum",
    "valueuom"
]

filtered_lab_chunks = []

chunk_number = 0

for chunk in pd.read_csv(
    "labevents.csv.gz",
    usecols=use_columns,
    chunksize=500000
):
    chunk_number += 1
    
    filtered_chunk = chunk[
        chunk["hadm_id"].isin(hf_hadm_ids) &
        chunk["itemid"].isin(selected_itemids                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       _set) &
        chunk["valuenum"].notna()
    ].copy()
    
    if len(filtered_chunk) > 0:
        filtered_lab_chunks.append(filtered_chunk)
    
    print("Finished chunk:", chunk_number, "| rows kept:", len(filtered_chunk))

hf_labs_raw = pd.concat(filtered_lab_chunks, ignore_index=True)

print("\nFinal extracted HF lab rows:", hf_labs_raw.shape)

hf_labs_raw.head()

In [ ]:
# Save the complete extracted heart-failure lab dataset

hf_labs_raw.to_pickle(
    "hf_labs_raw_complete.pkl.gz",
    compression="gzip"
)

print("Complete lab extraction saved successfully.")
print("Rows and columns:", hf_labs_raw.shape)

In [ ]:
from pathlib import Path

complete_file = Path("hf_labs_raw_complete.pkl.gz")

print("Complete saved file exists:", complete_file.exists())
print("Saved file size (MB):", round(complete_file.stat().st_size / (1024 * 1024), 2))

In [ ]:
# Keep only lab results from the first 24 hours after hospital admission

# Make sure times and admission IDs are in the correct format
hf_admissions["admittime"] = pd.to_datetime(
    hf_admissions["admittime"],
    errors="coerce"
)

hf_labs_raw["charttime"] = pd.to_datetime(
    hf_labs_raw["charttime"],
    errors="coerce"
)

hf_labs_raw["hadm_id"] = pd.to_numeric(
    hf_labs_raw["hadm_id"],
    errors="coerce"
).astype("Int64")

hf_admissions["hadm_id"] = pd.to_numeric(
    hf_admissions["hadm_id"],
    errors="coerce"
).astype("Int64")

# Create a fast admission-ID-to-admission-time lookup
admit_time_map = (
    hf_admissions
    .drop_duplicates(subset="hadm_id")
    .set_index("hadm_id")["admittime"]
)

# Add admission time to every extracted lab row
hf_labs_raw["admittime"] = hf_labs_raw["hadm_id"].map(admit_time_map)

# Calculate how many hours after admission each lab test happened
hf_labs_raw["hours_from_admission"] = (
    hf_labs_raw["charttime"] - hf_labs_raw["admittime"]
).dt.total_seconds() / 3600

# Keep only labs taken from 0 to 24 hours after admission
hf_labs_24h = hf_labs_raw.loc[
    hf_labs_raw["hours_from_admission"].between(0, 24),
    [
        "subject_id",
        "hadm_id",
        "itemid",
        "charttime",
        "valuenum",
        "valueuom",
        "hours_from_admission"
    ]
].copy()

print("Raw selected lab rows:", hf_labs_raw.shape[0])
print("First-24-hour lab rows:", hf_labs_24h.shape[0])
print("Admissions with at least one first-24-hour lab:",
      hf_labs_24h["hadm_id"].nunique())

hf_labs_24h.head()

In [ ]:
# Stage 2 checkpoint: save first-24-hour heart failure lab results

from pathlib import Path

hf_labs_24h.to_pickle(
    "hf_labs_first24h.pkl.gz",
    compression="gzip"
)

first24h_file = Path("hf_labs_first24h.pkl.gz")

print("First-24-hour lab file saved:", first24h_file.exists())
print("Rows saved:", hf_labs_24h.shape[0])
print("File size (MB):", round(first24h_file.stat().st_size / (1024 * 1024), 2))

In [ ]:
# Stage 3A: Label each laboratory item ID with its clinical biomarker name
# This makes the dataset understandable before we create one row per admission.

# Create a clear lookup between MIMIC item IDs and biomarker names
itemid_to_biomarker = {
    50862: "albumin",
    50882: "bicarbonate",
    50902: "chloride",
    50912: "creatinine",
    50931: "glucose",
    50963: "ntprobnp",
    50971: "potassium",
    50983: "sodium",
    51002: "troponin_i",
    51003: "troponin_t",
    51006: "urea_nitrogen_bun",
    51222: "hemoglobin",
    51265: "platelet_count",
    51300: "wbc_count"
}

# Make sure item IDs are stored as numbers
hf_labs_24h["itemid"] = pd.to_numeric(
    hf_labs_24h["itemid"],
    errors="coerce"
).astype("Int64")

# Add the clinical biomarker name beside every item ID
hf_labs_24h["biomarker"] = hf_labs_24h["itemid"].map(
    itemid_to_biomarker
)

# Check whether any selected lab rows failed to receive a biomarker name
print("Rows with no biomarker name:",
      hf_labs_24h["biomarker"].isna().sum())

# Summarise each biomarker and its measurement units
lab_audit = (
    hf_labs_24h
    .groupby(["biomarker", "itemid", "valueuom"])
    .agg(
        measurements=("valuenum", "size"),
        admissions=("hadm_id", "nunique"),
        minimum_value=("valuenum", "min"),
        median_value=("valuenum", "median"),
        maximum_value=("valuenum", "max")
    )
    .reset_index()
    .sort_values(["biomarker", "valueuom"])
)

display(lab_audit)

In [ ]:
# Stage 3B: calculate biomarker coverage across all heart-failure admissions

total_hf_admissions = hf_admissions["hadm_id"].nunique()

biomarker_coverage = (
    hf_labs_24h
    .groupby("biomarker")["hadm_id"]
    .nunique()
    .reset_index(name="admissions_with_test")
)

biomarker_coverage["total_hf_admissions"] = total_hf_admissions

biomarker_coverage["coverage_percent"] = (
    biomarker_coverage["admissions_with_test"]
    / biomarker_coverage["total_hf_admissions"]
    * 100
)

biomarker_coverage = biomarker_coverage.sort_values(
    "coverage_percent",
    ascending=False
)

display(biomarker_coverage)

In [ ]:
# Stage 3B: audit possible WBC and troponin item IDs in this MIMIC version

search_pattern = r"wbc|white blood|troponin"

wbc_troponin_audit = (
    d_labs[
        d_labs["label"].str.contains(
            search_pattern,
            case=False,
            na=False,
            regex=True
        )
    ][["itemid", "label", "fluid", "category"]]
    .sort_values(["label", "itemid"])
)

display(wbc_troponin_audit)

In [ ]:
# Stage 3B correction:
# Recover the standard blood WBC result and the additional Troponin I assay.
# This adds only missing biomarkers; it does not replace the saved dataset.

extra_itemid_to_biomarker = {
    51301: "wbc_count",
    52642: "troponin_i"
}

extra_itemids = set(extra_itemid_to_biomarker.keys())

# Keep one admission time for each hospital admission
admission_time_lookup = (
    hf_admissions[["hadm_id", "admittime"]]
    .dropna()
    .drop_duplicates(subset="hadm_id")
    .copy()
)

admission_time_lookup["hadm_id"] = pd.to_numeric(
    admission_time_lookup["hadm_id"],
    errors="coerce"
).astype("Int64")

admission_time_lookup["admittime"] = pd.to_datetime(
    admission_time_lookup["admittime"],
    errors="coerce"
)

hf_hadm_ids = set(
    admission_time_lookup["hadm_id"]
    .dropna()
    .astype(int)
)

extra_lab_chunks = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        "labevents.csv.gz",
        usecols=[
            "subject_id",
            "hadm_id",
            "itemid",
            "charttime",
            "valuenum",
            "valueuom"
        ],
        chunksize=500000
    ),
    start=1
):

    chunk["hadm_id"] = pd.to_numeric(
        chunk["hadm_id"],
        errors="coerce"
    ).astype("Int64")

    kept_rows = chunk[
        chunk["hadm_id"].isin(hf_hadm_ids)
        & chunk["itemid"].isin(extra_itemids)
        & chunk["valuenum"].notna()
    ].copy()

    if not kept_rows.empty:

        kept_rows["charttime"] = pd.to_datetime(
            kept_rows["charttime"],
            errors="coerce"
        )

        kept_rows = kept_rows.merge(
            admission_time_lookup,
            on="hadm_id",
            how="left"
        )

        kept_rows["hours_from_admission"] = (
            kept_rows["charttime"] - kept_rows["admittime"]
        ).dt.total_seconds() / 3600

        kept_rows = kept_rows[
            kept_rows["hours_from_admission"].between(0, 24)
        ].copy()

        extra_lab_chunks.append(
            kept_rows[
                [
                    "subject_id",
                    "hadm_id",
                    "itemid",
                    "charttime",
                    "valuenum",
                    "valueuom",
                    "hours_from_admission"
                ]
            ]
        )

    if chunk_number % 25 == 0:
        print("Finished chunk:", chunk_number)

hf_labs_extra = pd.concat(extra_lab_chunks, ignore_index=True)

hf_labs_extra["biomarker"] = hf_labs_extra["itemid"].map(
    extra_itemid_to_biomarker
)

print("Extra first-24-hour rows:", hf_labs_extra.shape[0])

extra_lab_audit = (
    hf_labs_extra
    .groupby(["biomarker", "itemid", "valueuom"])
    .agg(
        measurements=("valuenum", "size"),
        admissions=("hadm_id", "nunique"),
        minimum_value=("valuenum", "min"),
        median_value=("valuenum", "median"),
        maximum_value=("valuenum", "max")
    )
    .reset_index()
)

display(extra_lab_audit)

In [ ]:
# Stage 3C: build one corrected first-24-hour biomarker event table
# Replace old low-coverage WBC item 51300 with standard blood WBC item 51301.

# 1. Remove the old low-coverage WBC rows from the original table
hf_labs_without_old_wbc = hf_labs_24h.loc[
    hf_labs_24h["itemid"] != 51300
].copy()

# 2. Keep only the corrected standard blood WBC rows
hf_labs_extra_wbc = hf_labs_extra.loc[
    hf_labs_extra["itemid"] == 51301
].copy()

# 3. Create an updated dictionary of item IDs and clinical biomarker names
combined_itemid_to_biomarker = {
    itemid: biomarker
    for itemid, biomarker in itemid_to_biomarker.items()
    if itemid != 51300
}

combined_itemid_to_biomarker[51301] = "wbc_count"

# 4. Stack the original biomarker rows and corrected WBC rows together
hf_labs_24h_enriched = pd.concat(
    [hf_labs_without_old_wbc, hf_labs_extra_wbc],
    ignore_index=True
).copy()

# 5. Apply clear biomarker names to every row
hf_labs_24h_enriched["biomarker"] = (
    hf_labs_24h_enriched["itemid"]
    .map(combined_itemid_to_biomarker)
)

# 6. Remove only exact duplicate database records
rows_before = hf_labs_24h_enriched.shape[0]

hf_labs_24h_enriched = hf_labs_24h_enriched.drop_duplicates(
    subset=[
        "subject_id",
        "hadm_id",
        "itemid",
        "charttime",
        "valuenum",
        "valueuom"
    ]
).copy()

rows_after = hf_labs_24h_enriched.shape[0]

# 7. Confirm that corrected WBC is now included
wbc_admissions = hf_labs_24h_enriched.loc[
    hf_labs_24h_enriched["biomarker"] == "wbc_count",
    "hadm_id"
].nunique()

print("Rows before exact-duplicate removal:", rows_before)
print("Rows after exact-duplicate removal:", rows_after)
print("Admissions with corrected early WBC:", wbc_admissions)

# 8. Save the improved checkpoint
hf_labs_24h_enriched.to_pickle(
    "hf_labs_first24h_enriched.pkl.gz",
    compression="gzip"
)

print("Enriched first-24-hour lab checkpoint saved successfully.")

In [ ]:
# Stage 3D1: choose the earliest valid first-24-hour value
# for each biomarker in each heart-failure admission.

# Make a safe working copy of the enriched early-lab event table
hf_labs_for_baseline = hf_labs_24h_enriched.copy()

# Ensure dates and numeric lab values are in the correct format
hf_labs_for_baseline["charttime"] = pd.to_datetime(
    hf_labs_for_baseline["charttime"],
    errors="coerce"
)

hf_labs_for_baseline["valuenum"] = pd.to_numeric(
    hf_labs_for_baseline["valuenum"],
    errors="coerce"
)

# Keep only usable rows with an admission, biomarker name, time, and numeric result
hf_labs_for_baseline = hf_labs_for_baseline.dropna(
    subset=["hadm_id", "biomarker", "charttime", "valuenum"]
).copy()

# Put each admission's biomarker results in time order
hf_labs_for_baseline = hf_labs_for_baseline.sort_values(
    ["hadm_id", "biomarker", "charttime"]
)

# Keep the first available result for each biomarker in each admission
hf_labs_earliest = hf_labs_for_baseline.drop_duplicates(
    subset=["hadm_id", "biomarker"],
    keep="first"
).copy()

# Check what changed
print("Original enriched early-lab rows:", hf_labs_24h_enriched.shape[0])
print("Earliest-result rows:", hf_labs_earliest.shape[0])
print("Admissions represented:", hf_labs_earliest['hadm_id'].nunique())
print("Biomarkers represented:", hf_labs_earliest['biomarker'].nunique())

# Show a few selected baseline results
display(
    hf_labs_earliest[
        [
            "subject_id",
            "hadm_id",
            "biomarker",
            "itemid",
            "charttime",
            "valuenum",
            "valueuom",
            "hours_from_admission"
        ]
    ].head(15)
)

In [ ]:
# Stage 3D2: reshape earliest biomarker results into one row per admission
# Each biomarker becomes a separate column for later mortality prediction.

# Define the 13 biomarkers that are actually available in this project
expected_biomarkers = [
    "albumin",
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "ntprobnp",
    "platelet_count",
    "potassium",
    "sodium",
    "troponin_t",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Safety check: each admission should have only one chosen value per biomarker
duplicate_pairs = hf_labs_earliest.duplicated(
    subset=["hadm_id", "biomarker"]
).sum()

print("Duplicate admission-biomarker pairs:", duplicate_pairs)

# Turn the long table into a wide table:
# one row = one admission; one column = one biomarker
hf_biomarker_wide = (
    hf_labs_earliest
    .pivot(
        index=["subject_id", "hadm_id"],
        columns="biomarker",
        values="valuenum"
    )
    .reindex(columns=expected_biomarkers)
    .reset_index()
)

# Remove the extra column heading created by pandas
hf_biomarker_wide.columns.name = None

# Confirm the new table structure
print("Rows in wide biomarker table:", hf_biomarker_wide.shape[0])
print("Biomarker columns:", hf_biomarker_wide.shape[1] - 2)
print("Admissions represented:", hf_biomarker_wide["hadm_id"].nunique())

display(hf_biomarker_wide.head(10))

In [ ]:
# Stage 3E1: merge one-row-per-admission biomarkers with
# patient demographics and the in-hospital mortality outcome.

# Save the completed wide biomarker checkpoint
hf_biomarker_wide.to_pickle(
    "hf_biomarker_wide_earliest24h.pkl.gz",
    compression="gzip"
)

# Keep only columns needed for outcome checking, demographics,
# and later modelling decisions.
admission_columns = [
    "subject_id",
    "hadm_id",
    "admittime",
    "dischtime",
    "deathtime",
    "admission_type",
    "race",
    "gender",
    "age_at_admission",
    "hospital_mortality",
    "length_of_stay_days"
]

admission_info = hf_admissions[admission_columns].copy()
biomarker_info = hf_biomarker_wide.copy()

# Make sure the matching ID columns use the same numeric format
for id_column in ["subject_id", "hadm_id"]:
    admission_info[id_column] = pd.to_numeric(
        admission_info[id_column],
        errors="coerce"
    ).astype("Int64")

    biomarker_info[id_column] = pd.to_numeric(
        biomarker_info[id_column],
        errors="coerce"
    ).astype("Int64")

# Merge biomarkers with admission-level outcome and demographic information
hf_model_base = biomarker_info.merge(
    admission_info,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Confirm every biomarker admission matched to an HF admission record
print("Merge result:")
print(hf_model_base["_merge"].value_counts())

print("\nRows in merged modelling base:", hf_model_base.shape[0])
print(
    "Rows with missing mortality outcome:",
    hf_model_base["hospital_mortality"].isna().sum()
)

print("\nIn-hospital mortality counts:")
print(
    hf_model_base["hospital_mortality"]
    .value_counts(dropna=False)
    .sort_index()
)

# Remove the temporary merge-check column
hf_model_base = hf_model_base.drop(columns="_merge")

# Save this checkpoint before further cleaning
hf_model_base.to_pickle(
    "hf_model_base_before_cleaning.pkl.gz",
    compression="gzip"
)

display(
    hf_model_base[
        [
            "subject_id",
            "hadm_id",
            "age_at_admission",
            "gender",
            "hospital_mortality",
            "creatinine",
            "sodium",
            "wbc_count",
            "hemoglobin"
        ]
    ].head(10)
)

In [ ]:
# Stage 3E2: audit eligibility for a prediction made at 24 hours.
# Patients already dead or discharged at the 24-hour landmark are excluded.
hf_model_time_audit = hf_model_base.copy()
for time_column in ["admittime", "dischtime", "deathtime"]:
    hf_model_time_audit[time_column] = pd.to_datetime(
        hf_model_time_audit[time_column], errors="coerce"
    )

hf_model_time_audit["hours_until_discharge"] = (
    hf_model_time_audit["dischtime"] - hf_model_time_audit["admittime"]
).dt.total_seconds() / 3600
hf_model_time_audit["hours_until_death"] = (
    hf_model_time_audit["deathtime"] - hf_model_time_audit["admittime"]
).dt.total_seconds() / 3600

# Conservative landmark: stay >24 h, and no recorded in-hospital death <=24 h.
# For positive outcomes with no deathtime, dischtime is the fallback evidence
# that the patient was still hospitalised at the landmark.
hf_model_time_audit["death_by_24h"] = (
    (hf_model_time_audit["hospital_mortality"] == 1)
    & (hf_model_time_audit["hours_until_death"] <= 24)
)
hf_model_time_audit["eligible_at_24h"] = (
    (hf_model_time_audit["hours_until_discharge"] > 24)
    & ~hf_model_time_audit["death_by_24h"]
)

print("Admissions with early biomarkers:", len(hf_model_time_audit))
print("Admissions eligible at 24 h:", hf_model_time_audit["eligible_at_24h"].sum())
print("Admissions excluded:", (~hf_model_time_audit["eligible_at_24h"]).sum())
print("Recorded hospital deaths at or before 24 h:",
      hf_model_time_audit["death_by_24h"].sum())
# Row-level timestamp details are intentionally not displayed in this public version.


In [ ]:
# Stage 3E3: enforce 24-hour eligibility, not just report violations.
hf_model_24h_eligible = hf_model_time_audit.loc[
    hf_model_time_audit["eligible_at_24h"]
].copy()
hf_model_24h_eligible["hospital_mortality"] = (
    hf_model_24h_eligible["hospital_mortality"].astype("int8")
)

# Assertions prevent a later run from silently accepting an invalid cohort.
assert (hf_model_24h_eligible["hours_until_discharge"] > 24).all(), (
    "An admission not in hospital after 24 h was retained."
)
assert not hf_model_24h_eligible["death_by_24h"].any(), (
    "A patient recorded as dead by the landmark was retained."
)
assert not (
    (hf_model_24h_eligible["hospital_mortality"] == 1)
    & (hf_model_24h_eligible["hours_until_death"] <= 24)
).any(), "Death at or before 24 h was retained."
assert hf_model_24h_eligible["hadm_id"].is_unique

print("Eligible admissions:", len(hf_model_24h_eligible))
print("Deaths after the 24-hour landmark:",
      int(hf_model_24h_eligible["hospital_mortality"].sum()))
# This checkpoint contains restricted admission-level MIMIC data: DO NOT UPLOAD.
hf_model_24h_eligible.to_pickle(
    "hf_model_24h_eligible_before_cleaning.pkl.gz", compression="gzip"
)


In [ ]:
# Stage 3F1: validate the 24-hour cohort and audit biomarker missingness.
# Do not fill missing values yet. First must understand what is missing.

# The 13 biomarkers currently available in the first-24-hour dataset
biomarker_columns = [
    "albumin",
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "ntprobnp",
    "platelet_count",
    "potassium",
    "sodium",
    "troponin_t",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Safety check: confirm every expected biomarker column exists
missing_columns = [
    column for column in biomarker_columns
    if column not in hf_model_24h_eligible.columns
]

if missing_columns:
    raise ValueError(
        f"Expected biomarker columns are missing: {missing_columns}"
    )

# Check whether the 24-hour eligibility rule left any timing problems
timing_audit = pd.Series({
    "eligible_admissions": hf_model_24h_eligible.shape[0],
    "missing_hours_until_discharge":
        hf_model_24h_eligible["hours_until_discharge"].isna().sum(),
    "negative_hours_until_discharge":
        (hf_model_24h_eligible["hours_until_discharge"] < 0).sum(),
    "retained_admissions_at_or_under_24h":
        (hf_model_24h_eligible["hours_until_discharge"] <= 24).sum(),
    "retained_deaths_before_24h":
        (
            (hf_model_24h_eligible["hospital_mortality"] == 1)
            & (hf_model_24h_eligible["hours_until_death"] <= 24)
        ).sum()
})

print("TIMING VALIDATION")
print(timing_audit)

# Count available and missing results for every biomarker
missingness_overall = pd.DataFrame({
    "available_admissions":
        hf_model_24h_eligible[biomarker_columns].notna().sum(),
    "missing_admissions":
        hf_model_24h_eligible[biomarker_columns].isna().sum()
})

missingness_overall["available_percent"] = (
    missingness_overall["available_admissions"]
    / hf_model_24h_eligible.shape[0]
    * 100
).round(2)

missingness_overall["missing_percent"] = (
    missingness_overall["missing_admissions"]
    / hf_model_24h_eligible.shape[0]
    * 100
).round(2)

missingness_overall = missingness_overall.sort_values(
    "available_percent",
    ascending=False
)

# Compare missingness between survivors and patients who died
survivors = hf_model_24h_eligible.loc[
    hf_model_24h_eligible["hospital_mortality"] == 0,
    biomarker_columns
]

deaths = hf_model_24h_eligible.loc[
    hf_model_24h_eligible["hospital_mortality"] == 1,
    biomarker_columns
]

missingness_by_outcome = pd.DataFrame({
    "missing_percent_survived": (
        survivors.isna().mean() * 100
    ).round(2),
    "missing_percent_died": (
        deaths.isna().mean() * 100
    ).round(2)
})

missingness_by_outcome["difference_died_minus_survived"] = (
    missingness_by_outcome["missing_percent_died"]
    - missingness_by_outcome["missing_percent_survived"]
).round(2)

missingness_by_outcome = missingness_by_outcome.sort_values(
    "difference_died_minus_survived",
    ascending=False
)

print("\nOVERALL BIOMARKER AVAILABILITY")
display(missingness_overall)

print("\nMISSINGNESS BY MORTALITY OUTCOME")
display(missingness_by_outcome)

# Save these audit tables for your later Results/Methods documentation
missingness_overall.to_csv(
    "biomarker_missingness_overall_24h.csv"
)

missingness_by_outcome.to_csv(
    "biomarker_missingness_by_outcome_24h.csv"
)

print("\nMissingness audit files saved successfully.")

In [ ]:
# Stage 3F2: define the primary and secondary biomarker strategies.
# This does not change patient data or train a model.
# 1. Biomarkers suitable for the main routine-model pathway.
# These were available for roughly 96–98% of eligible admissions.
core_biomarkers = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# 2. Clinically important but selectively measured biomarkers.
# These will not be forced into the main model.
selective_biomarkers = [
    "albumin",
    "troponin_t",
    "ntprobnp"
]

# 3. Check that all expected columns exist before continuing.
required_columns = (
    core_biomarkers
    + selective_biomarkers
    + ["hospital_mortality"]
)

missing_columns = [
    column for column in required_columns
    if column not in hf_model_24h_eligible.columns
]

if missing_columns:
    raise ValueError(
        f"These expected columns are missing: {missing_columns}"
    )

# 4. Record the planned role of every available biomarker.
feature_plan = pd.DataFrame({
    "biomarker": core_biomarkers + selective_biomarkers,
    "analysis_role": (
        ["primary_routine_model"] * len(core_biomarkers)
        + ["secondary_exploratory_analysis"] * len(selective_biomarkers)
    ),
    "reason": (
        ["high early-test availability"] * len(core_biomarkers)
        + ["clinically relevant but selectively measured"] * len(selective_biomarkers)
    )
})

print("FEATURE STRATEGY")
display(feature_plan)

# 5. Audit how many admissions would remain if every listed biomarker
# had to be present. This is an audit only;  not deleting rows.
core_complete = hf_model_24h_eligible[
    core_biomarkers
].notna().all(axis=1)

core_plus_albumin_complete = (
    core_complete
    & hf_model_24h_eligible["albumin"].notna()
)

core_plus_troponin_t_complete = (
    core_complete
    & hf_model_24h_eligible["troponin_t"].notna()
)

core_plus_ntprobnp_complete = (
    core_complete
    & hf_model_24h_eligible["ntprobnp"].notna()
)

all_selective_complete = (
    core_complete
    & hf_model_24h_eligible[selective_biomarkers].notna().all(axis=1)
)

feature_set_audit = pd.DataFrame({
    "analysis_path": [
        "Core routine biomarkers only",
        "Core routine biomarkers + albumin",
        "Core routine biomarkers + troponin T",
        "Core routine biomarkers + NT-proBNP",
        "Core routine biomarkers + all selective biomarkers"
    ],
    "admissions_with_all_required_tests": [
        core_complete.sum(),
        core_plus_albumin_complete.sum(),
        core_plus_troponin_t_complete.sum(),
        core_plus_ntprobnp_complete.sum(),
        all_selective_complete.sum()
    ],
    "percent_of_eligible_cohort": [
        round(core_complete.mean() * 100, 2),
        round(core_plus_albumin_complete.mean() * 100, 2),
        round(core_plus_troponin_t_complete.mean() * 100, 2),
        round(core_plus_ntprobnp_complete.mean() * 100, 2),
        round(all_selective_complete.mean() * 100, 2)
    ],
    "later_in_hospital_deaths": [
        hf_model_24h_eligible.loc[core_complete, "hospital_mortality"].sum(),
        hf_model_24h_eligible.loc[
            core_plus_albumin_complete, "hospital_mortality"
        ].sum(),
        hf_model_24h_eligible.loc[
            core_plus_troponin_t_complete, "hospital_mortality"
        ].sum(),
        hf_model_24h_eligible.loc[
            core_plus_ntprobnp_complete, "hospital_mortality"
        ].sum(),
        hf_model_24h_eligible.loc[
            all_selective_complete, "hospital_mortality"
        ].sum()
    ]
})

print("\nCOMPLETE-CASE FEATURE-SET AUDIT")
display(feature_set_audit)

# 6. Save this decision and audit for later Methods/Results writing.
feature_plan.to_csv(
    "feature_strategy_24h.csv",
    index=False
)

feature_set_audit.to_csv(
    "feature_set_complete_case_audit_24h.csv",
    index=False
)

print("\nFeature strategy audit files saved successfully.")

In [ ]:
# Stage 3F3: audit units and extreme values for the core routine biomarkers.
# This is an audit only. It does not delete or change any patient values.

# Define the core biomarkers used in the primary mortality-model pathway.
core_biomarkers = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Keep the earliest lab-event records for core biomarkers.
# This table still contains the original recorded laboratory units.
core_baseline_events = hf_labs_earliest.loc[
    hf_labs_earliest["biomarker"].isin(core_biomarkers),
    [
        "subject_id",
        "hadm_id",
        "biomarker",
        "valuenum",
        "valueuom",
        "charttime",
        "hours_from_admission"
    ]
].copy()

# Make missing or blank units visible instead of hiding them.
core_baseline_events["valueuom"] = (
    core_baseline_events["valueuom"]
    .fillna("MISSING_UNIT")
    .astype(str)
    .str.strip()
    .replace("", "MISSING_UNIT")
)

# Count each unit used for each biomarker and summarise its values.
unit_audit = (
    core_baseline_events
    .groupby(["biomarker", "valueuom"], dropna=False)
    .agg(
        measurements=("valuenum", "size"),
        admissions=("hadm_id", "nunique"),
        minimum_value=("valuenum", "min"),
        median_value=("valuenum", "median"),
        maximum_value=("valuenum", "max")
    )
    .reset_index()
    .sort_values(["biomarker", "measurements"], ascending=[True, False])
)

# Summarise whether each biomarker has one consistent unit or multiple units.
dominant_units = (
    unit_audit
    .drop_duplicates(subset="biomarker", keep="first")
    [["biomarker", "valueuom", "measurements"]]
    .rename(
        columns={
            "valueuom": "most_common_unit",
            "measurements": "measurements_in_most_common_unit"
        }
    )
)

unit_summary = (
    core_baseline_events
    .groupby("biomarker")
    .agg(
        admissions=("hadm_id", "nunique"),
        units_observed=("valueuom", "nunique"),
        missing_unit_rows=(
            "valueuom",
            lambda values: (values == "MISSING_UNIT").sum()
        )
    )
    .reset_index()
    .merge(dominant_units, on="biomarker", how="left")
    .sort_values("biomarker")
)

# Create a descriptive-value audit from the final 24-hour eligible cohort.
quantiles = (
    hf_model_24h_eligible[core_biomarkers]
    .quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)

quantiles.columns = [
    "p01",
    "p05",
    "p25",
    "median",
    "p75",
    "p95",
    "p99"
]

value_audit = (
    hf_model_24h_eligible[core_biomarkers]
    .agg(["count", "min", "max"])
    .T
    .join(quantiles)
    .reset_index()
    .rename(columns={"index": "biomarker"})
)

# Flag only very extreme values using a broad robust IQR rule.
# A flag means "inspect later"; it does not mean "delete".
q1 = hf_model_24h_eligible[core_biomarkers].quantile(0.25)
q3 = hf_model_24h_eligible[core_biomarkers].quantile(0.75)
iqr = q3 - q1

lower_flags = q1 - (3 * iqr)
upper_flags = q3 + (3 * iqr)

outlier_rows = []

for biomarker in core_biomarkers:
    values = hf_model_24h_eligible[biomarker].dropna()

    flagged = (
        (values < lower_flags[biomarker])
        | (values > upper_flags[biomarker])
    )

    outlier_rows.append({
        "biomarker": biomarker,
        "lower_flag_threshold": round(lower_flags[biomarker], 3),
        "upper_flag_threshold": round(upper_flags[biomarker], 3),
        "flagged_values": int(flagged.sum()),
        "flagged_percent": round(flagged.mean() * 100, 3)
    })

outlier_audit = pd.DataFrame(outlier_rows).sort_values(
    "flagged_percent",
    ascending=False
)

print("UNIT SUMMARY")
display(unit_summary)

print("\nUNIT DETAILS")
display(unit_audit)

print("\nVALUE SUMMARY")
display(value_audit)

print("\nVERY EXTREME VALUE FLAGS")
display(outlier_audit)

# Save audit files for the Methods and Results sections later.
unit_summary.to_csv("core_biomarker_unit_summary_24h.csv", index=False)
unit_audit.to_csv("core_biomarker_unit_details_24h.csv", index=False)
value_audit.to_csv("core_biomarker_value_summary_24h.csv", index=False)
outlier_audit.to_csv("core_biomarker_extreme_value_flags_24h.csv", index=False)

print("\nUnit and value audit files saved successfully.")

In [ ]:
# Stage 3F3B: inspect actual extreme records without changing any values.
# IQR flags identify unusual values, but this step lets review them before
# deciding whether any value is likely to be a true clinical result or data problem.

# Keep only biomarkers with at least one statistical extreme-value flag
flagged_biomarkers = outlier_audit.loc[
    outlier_audit["flagged_values"] > 0,
    "biomarker"
].tolist()

# Columns that help me understand each extreme result in clinical context
context_columns = [
    "subject_id",
    "hadm_id",
    "age_at_admission",
    "gender",
    "hospital_mortality",
    "hours_until_discharge"
]

# Create a dictionary so each biomarker is shown with its recorded unit
unit_map = unit_summary.set_index(
    "biomarker"
)["most_common_unit"].to_dict()

# Store the lowest and highest records for every flagged biomarker
extreme_review_frames = []

for biomarker in flagged_biomarkers:

    # Keep admissions with a real value for this biomarker
    biomarker_rows = hf_model_24h_eligible[
        context_columns + [biomarker]
    ].dropna(subset=[biomarker]).copy()

    # Select the three lowest values
    lowest_values = biomarker_rows.nsmallest(
        3,
        biomarker
    ).copy()

    lowest_values["biomarker"] = biomarker
    lowest_values["tail"] = "lowest 3"

    # Select the three highest values
    highest_values = biomarker_rows.nlargest(
        3,
        biomarker
    ).copy()

    highest_values["biomarker"] = biomarker
    highest_values["tail"] = "highest 3"

    # Give both tables one common value column
    lowest_values = lowest_values.rename(
        columns={biomarker: "value"}
    )

    highest_values = highest_values.rename(
        columns={biomarker: "value"}
    )

    # Add these results to the full review table
    extreme_review_frames.extend(
        [lowest_values, highest_values]
    )

# Combine all reviewed extreme records into one table
extreme_value_review = pd.concat(
    extreme_review_frames,
    ignore_index=True
)

# Add the laboratory unit beside every value
extreme_value_review["unit"] = extreme_value_review[
    "biomarker"
].map(unit_map)

# Put the columns in a clear clinical-review order
extreme_value_review = extreme_value_review[
    [
        "biomarker",
        "tail",
        "value",
        "unit",
        "subject_id",
        "hadm_id",
        "age_at_admission",
        "gender",
        "hospital_mortality",
        "hours_until_discharge"
    ]
].sort_values(
    ["biomarker", "tail", "value"]
)

print("Biomarkers requiring extreme-value review:")
print(flagged_biomarkers)

print("\nExtreme records shown:")
print(extreme_value_review.shape[0])

display(extreme_value_review)

# Save the review table for my project audit trail
extreme_value_review.to_csv(
    "core_biomarker_extreme_value_record_review_24h.csv",
    index=False
)

print("\nExtreme-value record review saved successfully.")

In [ ]:
# Stage 3F4: lock the raw primary dataset for descriptive analysis and modelling.
# No values are deleted, imputed, scaled, transformed, or clipped here.

# Predictors known within the first 24 hours of admission
primary_predictors = [
    "age_at_admission",
    "gender",
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Identifiers are retained for auditing but will not be model predictors
id_columns = [
    "subject_id",
    "hadm_id"
]

# The outcome the future model will learn to predict
target_column = "hospital_mortality"

# Confirm every required column exists before creating the dataset
required_columns = id_columns + primary_predictors + [target_column]

missing_columns = [
    column for column in required_columns
    if column not in hf_model_24h_eligible.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

# Create the locked raw primary dataset
hf_primary_model_raw = hf_model_24h_eligible[
    required_columns
].copy()

# Safety check: one row must equal one hospital admission
duplicate_admissions = hf_primary_model_raw[
    "hadm_id"
].duplicated().sum()

if duplicate_admissions != 0:
    raise ValueError(
        f"Duplicate admission rows found: {duplicate_admissions}"
    )

# Audit missing values, but do not fill them yet
primary_missingness = pd.DataFrame({
    "missing_values": hf_primary_model_raw[
        primary_predictors
    ].isna().sum()
})

primary_missingness["missing_percent"] = (
    primary_missingness["missing_values"]
    / hf_primary_model_raw.shape[0]
    * 100
).round(2)

print("PRIMARY RAW MODELLING DATASET")
print("Rows:", hf_primary_model_raw.shape[0])
print("Unique admissions:", hf_primary_model_raw["hadm_id"].nunique())
print("Duplicate admission rows:", duplicate_admissions)

print("\nMortality outcome:")
print(
    hf_primary_model_raw[target_column]
    .value_counts()
    .sort_index()
)

print("\nGender values:")
print(
    hf_primary_model_raw["gender"]
    .value_counts(dropna=False)
)

print("\nMissingness in primary predictors:")
display(primary_missingness.sort_values(
    "missing_percent",
    ascending=False
))

# Save the untouched raw modelling dataset as a checkpoint
hf_primary_model_raw.to_pickle(
    "hf_primary_model_raw_24h.pkl.gz",
    compression="gzip"
)

print("\nRaw primary modelling dataset saved successfully.")

display(hf_primary_model_raw.head(10))

In [ ]:
# Stage 4A: create a cohort-flow table and first descriptive figure.
# This does not change patient data and does not train a model.

# Count unique hospital admissions at each key project stage
all_hf_admissions = heart_failure_diagnoses["hadm_id"].nunique()

admissions_with_early_biomarkers = (
    hf_model_time_audit["hadm_id"].nunique()
)

admissions_eligible_at_24h = (
    hf_model_24h_eligible["hadm_id"].nunique()
)

primary_raw_modelling_admissions = (
    hf_primary_model_raw["hadm_id"].nunique()
)

# Build a clear table showing how the cohort was constructed
cohort_flow = pd.DataFrame({
    "cohort_stage": [
        "All unique heart-failure admissions",
        "Admissions with selected early biomarkers",
        "Admissions still in hospital at 24 hours",
        "Primary raw modelling dataset"
    ],
    "admissions": [
        all_hf_admissions,
        admissions_with_early_biomarkers,
        admissions_eligible_at_24h,
        primary_raw_modelling_admissions
    ]
})

# Show the percentage retained compared with all heart-failure admissions
cohort_flow["percent_of_all_hf_admissions"] = (
    cohort_flow["admissions"]
    / all_hf_admissions
    * 100
).round(2)

print("COHORT FLOW SUMMARY")
display(cohort_flow)

# Create the first proper project graph
plt.figure(figsize=(11, 5))

bars = plt.barh(
    cohort_flow["cohort_stage"],
    cohort_flow["admissions"]
)

plt.xlabel("Number of hospital admissions")
plt.title("Cohort flow for the 24-hour heart-failure mortality study")

# Put the first stage at the top of the chart
plt.gca().invert_yaxis()

# Add admission counts and retained percentages beside each bar
plt.xlim(0, cohort_flow["admissions"].max() * 1.18)

for bar, admissions, percentage in zip(
    bars,
    cohort_flow["admissions"],
    cohort_flow["percent_of_all_hf_admissions"]
):
    plt.text(
        bar.get_width() + (all_hf_admissions * 0.01),
        bar.get_y() + bar.get_height() / 2,
        f"{admissions:,} ({percentage:.1f}%)",
        va="center"
    )

plt.tight_layout()

# Save the chart and table for my report later
plt.savefig(
    "figure_1_cohort_flow_24h.png",
    dpi=300,
    bbox_inches="tight"
)

cohort_flow.to_csv(
    "cohort_flow_summary_24h.csv",
    index=False
)

plt.show()

print("\nCohort-flow figure and summary table saved successfully.")

In [ ]:
# Stage 4B: describe the mortality outcome in the final modelling cohort.
# This creates a chart only. It does not change data or train a model.

# Count survivors (0) and later in-hospital deaths (1).
outcome_counts = (
    hf_primary_model_raw["hospital_mortality"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

# Give the outcome values clear labels for the table and chart.
outcome_summary = pd.DataFrame({
    "outcome": [
        "Survived to hospital discharge",
        "Died later in hospital"
    ],
    "admissions": outcome_counts.values
})

# Calculate the percentage of the final eligible cohort in each outcome group.
outcome_summary["percent"] = (
    outcome_summary["admissions"]
    / outcome_summary["admissions"].sum()
    * 100
).round(2)

print("MORTALITY OUTCOME SUMMARY")
display(outcome_summary)

# Create the mortality outcome bar chart.
plt.figure(figsize=(8, 5))

bars = plt.bar(
    outcome_summary["outcome"],
    outcome_summary["admissions"]
)

plt.ylabel("Number of hospital admissions")
plt.title("Later in-hospital mortality in the 24-hour eligible heart-failure cohort")

# Add the count and percentage above each bar.
for bar, admissions, percentage in zip(
    bars,
    outcome_summary["admissions"],
    outcome_summary["percent"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{admissions:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom"
    )

plt.ylim(0, outcome_summary["admissions"].max() * 1.12)
plt.tight_layout()

# Save the figure and table for report later.
plt.savefig(
    "figure_2_mortality_outcome_24h.png",
    dpi=300,
    bbox_inches="tight"
)

outcome_summary.to_csv(
    "mortality_outcome_summary_24h.csv",
    index=False
)

plt.show()

print("\nMortality figure and summary table saved successfully.")

In [ ]:
# Stage 4C: show missingness for the 10 routine biomarkers.
# This graph explains why these biomarkers are suitable for the primary model.

# Keep only the 10 routine laboratory biomarkers.
core_biomarkers = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Calculate how often each routine biomarker is missing.
missingness_plot = pd.DataFrame({
    "missing_percent": (
        hf_primary_model_raw[core_biomarkers]
        .isna()
        .mean()
        * 100
    )
})

# Put the most-missing biomarker at the top of the chart.
missingness_plot = missingness_plot.sort_values(
    "missing_percent",
    ascending=False
)

print("ROUTINE BIOMARKER MISSINGNESS")
display(missingness_plot.round(2))

# Create the graph.
plt.figure(figsize=(10, 5))

bars = plt.barh(
    missingness_plot.index,
    missingness_plot["missing_percent"]
)

plt.xlabel("Missing values (%)")
plt.title(
    "Missingness of routine biomarkers in the 24-hour heart-failure cohort"
)

plt.gca().invert_yaxis()

# Add the percentage beside each bar.
plt.xlim(0, missingness_plot["missing_percent"].max() + 0.8)

for bar, percentage in zip(
    bars,
    missingness_plot["missing_percent"]
):
    plt.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{percentage:.2f}%",
        va="center"
    )

plt.tight_layout()

# Save the figure and its summary table.
plt.savefig(
    "figure_3_routine_biomarker_missingness_24h.png",
    dpi=300,
    bbox_inches="tight"
)

missingness_plot.to_csv(
    "routine_biomarker_missingness_24h.csv"
)

plt.show()

print("\nMissingness figure and summary table saved successfully.")

In [ ]:
# Stage 4D1: descriptive biomarker summary by later in-hospital mortality.
# This creates a table only. It does not alter values or train a model.

core_biomarkers = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Give the outcome groups readable names.
descriptive_data = hf_primary_model_raw.copy()

descriptive_data["outcome_group"] = descriptive_data[
    "hospital_mortality"
].map({
    0: "Survived",
    1: "Died later in hospital"
})

# Create one summary row for each outcome group and biomarker.
biomarker_outcome_summary = (
    descriptive_data
    .melt(
        id_vars="outcome_group",
        value_vars=core_biomarkers,
        var_name="biomarker",
        value_name="value"
    )
    .dropna(subset=["value"])
    .groupby(["biomarker", "outcome_group"])
    .agg(
        admissions_with_test=("value", "count"),
        median=("value", "median"),
        p25=("value", lambda x: x.quantile(0.25)),
        p75=("value", lambda x: x.quantile(0.75)),
        mean=("value", "mean")
    )
    .reset_index()
)

# Convert from long format to a readable comparison table.
biomarker_outcome_table = (
    biomarker_outcome_summary
    .pivot(
        index="biomarker",
        columns="outcome_group",
        values=["admissions_with_test", "median", "p25", "p75", "mean"]
    )
)

# Make the column names clear.
biomarker_outcome_table.columns = [
    f"{statistic}_{group.replace(' ', '_').lower()}"
    for statistic, group in biomarker_outcome_table.columns
]

biomarker_outcome_table = (
    biomarker_outcome_table
    .reset_index()
    .round(2)
)

print("BIOMARKER SUMMARY BY MORTALITY OUTCOME")
display(biomarker_outcome_table)

# Save the report-ready descriptive table.
biomarker_outcome_table.to_csv(
    "biomarker_summary_by_mortality_24h.csv",
    index=False
)

print("\nBiomarker outcome summary table saved successfully.")

In [ ]:
# Stage 4D2: compare early creatinine values between survivors and deaths.
# This is a visualisation only. It does not remove or modify any data.

creatinine_plot_data = hf_primary_model_raw[
    ["hospital_mortality", "creatinine"]
].dropna().copy()

# Make the outcome labels readable on the graph.
creatinine_plot_data["outcome_group"] = (
    creatinine_plot_data["hospital_mortality"]
    .map({
        0: "Survived",
        1: "Died later in hospital"
    })
)

# Calculate a display range that protects the main plot from extreme values.
# The full data remain unchanged.
creatinine_lower_display = creatinine_plot_data["creatinine"].quantile(0.01)
creatinine_upper_display = creatinine_plot_data["creatinine"].quantile(0.99)

print("Creatinine values used in the plot:", creatinine_plot_data.shape[0])
print("Display range:", round(creatinine_lower_display, 2), "to",
      round(creatinine_upper_display, 2), "mg/dL")

plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        creatinine_plot_data.loc[
            creatinine_plot_data["outcome_group"] == "Survived",
            "creatinine"
        ],
        creatinine_plot_data.loc[
            creatinine_plot_data["outcome_group"] == "Died later in hospital",
            "creatinine"
        ]
    ],
    labels=["Survived", "Died later in hospital"],
    showfliers=False
)

plt.ylabel("Early creatinine (mg/dL)")
plt.title("Early creatinine by later in-hospital mortality")

plt.ylim(
    creatinine_lower_display,
    creatinine_upper_display
)

plt.tight_layout()

plt.savefig(
    "figure_4_creatinine_by_mortality_24h.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nCreatinine comparison figure saved successfully.")

In [ ]:
# Stage 4D3: compare early BUN values between survivors and deaths.
# This creates one graph only. It does not change or remove values from the dataset.

# Keep the outcome and early BUN value needed for this chart.
bun_plot_data = hf_primary_model_raw[
    ["hospital_mortality", "urea_nitrogen_bun"]
].dropna().copy()

# Give the mortality outcome readable names for the graph.
bun_plot_data["outcome_group"] = bun_plot_data[
    "hospital_mortality"
].map({
    0: "Survived",
    1: "Died later in hospital"
})

# Find the central 98% of BUN values for a readable display range.
# Extreme values remain in the original dataset.
bun_lower_display = bun_plot_data[
    "urea_nitrogen_bun"
].quantile(0.01)

bun_upper_display = bun_plot_data[
    "urea_nitrogen_bun"
].quantile(0.99)

print("BUN values used in the plot:", bun_plot_data.shape[0])

print(
    "Display range:",
    round(bun_lower_display, 2),
    "to",
    round(bun_upper_display, 2),
    "mg/dL"
)

# Start a new figure.
plt.figure(figsize=(8, 5))

# Draw one BUN boxplot for survivors and one for deaths.
plt.boxplot(
    [
        bun_plot_data.loc[
            bun_plot_data["outcome_group"] == "Survived",
            "urea_nitrogen_bun"
        ],
        bun_plot_data.loc[
            bun_plot_data["outcome_group"] == "Died later in hospital",
            "urea_nitrogen_bun"
        ]
    ],
    labels=[
        "Survived",
        "Died later in hospital"
    ],
    showfliers=False
)

# Add clear labels.
plt.ylabel("Early blood urea nitrogen / BUN (mg/dL)")
plt.title("Early BUN by later in-hospital mortality")

# Keep the graph readable despite very high BUN values.
plt.ylim(
    bun_lower_display,
    bun_upper_display
)

plt.tight_layout()

# Save a report-quality image.
plt.savefig(
    "figure_5_bun_by_mortality_24h.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nBUN comparison figure saved successfully.")

In [ ]:
# Stage 4D4: compare early sodium values between survivors and deaths.
# This creates one graph only. It does not change the main dataset.

# Take only mortality outcome and sodium for this graph.
sodium_plot_data = hf_primary_model_raw[
    ["hospital_mortality", "sodium"]
].dropna().copy()

# Turn 0 and 1 into readable labels.
sodium_plot_data["outcome_group"] = sodium_plot_data[
    "hospital_mortality"
].map({
    0: "Survived",
    1: "Died later in hospital"
})

# Set a readable visible range using the 1st and 99th percentiles.
# No patient values are removed.
sodium_lower_display = sodium_plot_data["sodium"].quantile(0.01)
sodium_upper_display = sodium_plot_data["sodium"].quantile(0.99)

print("Sodium values used in the plot:", sodium_plot_data.shape[0])
print(
    "Display range:",
    round(sodium_lower_display, 1),
    "to",
    round(sodium_upper_display, 1),
    "mEq/L"
)

# Start the figure.
plt.figure(figsize=(8, 5))

# Draw one boxplot for each mortality group.
plt.boxplot(
    [
        sodium_plot_data.loc[
            sodium_plot_data["outcome_group"] == "Survived",
            "sodium"
        ],
        sodium_plot_data.loc[
            sodium_plot_data["outcome_group"] == "Died later in hospital",
            "sodium"
        ]
    ],
    labels=["Survived", "Died later in hospital"],
    showfliers=False
)

# Label the graph clearly.
plt.ylabel("Early sodium (mEq/L)")
plt.title("Early sodium by later in-hospital mortality")

# Apply the readable visible range.
plt.ylim(
    sodium_lower_display,
    sodium_upper_display
)

plt.tight_layout()

# Save a report-quality copy.
plt.savefig(
    "figure_6_sodium_by_mortality_24h.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSodium comparison figure saved successfully.")

In [ ]:
# Stage 4E: correlation heatmap for routine early biomarkers.
# Keep the 10 routine biomarkers used in the primary model.
core_biomarkers = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Create a table containing only these biomarker columns.
correlation_data = hf_primary_model_raw[
    core_biomarkers
].copy()

# Calculate Spearman correlations.
# Spearman is useful here because several biomarkers are skewed
# and may contain clinically real extreme values.
correlation_matrix = correlation_data.corr(
    method="spearman",
    min_periods=1000
)

print("SPEARMAN CORRELATION MATRIX")
display(correlation_matrix.round(2))

# Create the heatmap figure.
plt.figure(figsize=(11, 9))

heatmap_image = plt.imshow(
    correlation_matrix,
    vmin=-1,
    vmax=1
)

plt.colorbar(
    heatmap_image,
    label="Spearman correlation"
)

plt.xticks(
    range(len(core_biomarkers)),
    core_biomarkers,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(core_biomarkers)),
    core_biomarkers
)

plt.title(
    "Correlation between routine early biomarkers in heart-failure admissions"
)

plt.tight_layout()

# Save the figure and correlation table for the report.
plt.savefig(
    "figure_7_core_biomarker_correlation_heatmap_24h.png",
    dpi=300,
    bbox_inches="tight"
)

correlation_matrix.to_csv(
    "core_biomarker_spearman_correlation_24h.csv"
)

plt.show()

print("\nCorrelation heatmap and table saved successfully.")

In [ ]:
# Stage 4F: Mortality percentage by age group

# Create age-group categories from continuous age values
hf_primary_model_raw["age_group"] = pd.cut(
    hf_primary_model_raw["age_at_admission"],
    bins=[0, 50, 60, 70, 80, 120],
    labels=["<50", "50-59", "60-69", "70-79", "80+"],
    right=False
)

# Calculate mortality percentage within each age group
age_mortality = (
    hf_primary_model_raw
    .groupby("age_group", observed=False)["hospital_mortality"]
    .mean()
    .mul(100)
    .reset_index()
)

print("Mortality percentage by age group")
display(age_mortality)

# Create line chart
plt.figure(figsize=(8, 5))

plt.plot(
    age_mortality["age_group"],
    age_mortality["hospital_mortality"],
    marker="o"
)

plt.title("Later In-Hospital Mortality by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Mortality (%)")
plt.grid(True)

plt.tight_layout()
plt.show()
print("\nAge-group mortality figure and table saved successfully.")

In [ ]:
# Stage 4G: Mortality percentage by sex

# Calculate mortality percentage by sex
sex_mortality = (
    hf_primary_model_raw
    .groupby("gender")["hospital_mortality"]
    .mean()
    .mul(100)
    .reset_index()
)

print("Mortality percentage by sex")
display(sex_mortality)

# Create bar chart
plt.figure(figsize=(6, 5))

plt.bar(
    sex_mortality["gender"],
    sex_mortality["hospital_mortality"]
)

plt.title("Later In-Hospital Mortality by Sex")
plt.xlabel("Sex")
plt.ylabel("Mortality (%)")

plt.tight_layout()
plt.show()
print("\nSex mortality figure and table saved successfully.")

In [ ]:
# Stage 4H: Final cohort descriptive summary 
# Demographics
n_total = hf_primary_model_raw.shape[0]
n_male = (hf_primary_model_raw["gender"] == "M").sum()
n_female = (hf_primary_model_raw["gender"] == "F").sum()

print("COHORT SIZE AND SEX DISTRIBUTION")
print("Total cohort:", n_total)
print(f"Male:   {n_male} ({round(n_male/n_total*100, 1)}%)")
print(f"Female: {n_female} ({round(n_female/n_total*100, 1)}%)")

# Age and biomarker descriptive statistics
summary_columns = [
    "age_at_admission",
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

descriptive_table = pd.DataFrame({
    "n_available": hf_primary_model_raw[summary_columns].notna().sum(),
    "mean": hf_primary_model_raw[summary_columns].mean().round(2),
    "sd": hf_primary_model_raw[summary_columns].std().round(2),
    "median": hf_primary_model_raw[summary_columns].median().round(2),
    "q1": hf_primary_model_raw[summary_columns].quantile(0.25).round(2),
    "q3": hf_primary_model_raw[summary_columns].quantile(0.75).round(2)
})

print("\nAGE AND BIOMARKER DESCRIPTIVE STATISTICS (raw, non-imputed values)")
display(descriptive_table)

# Save for your dissertation records
descriptive_table.to_csv("table_4_1_cohort_descriptives.csv")

print("\nStage 4H cohort descriptive summary completed successfully.")

In [ ]:
# Stage 5A: Define predictors (X) and outcome (y)
# Variables intended for the primary mortality model
model_features = [
    "age_at_admission",
    "gender",
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Create predictor matrix
X = hf_primary_model_raw[model_features].copy()

# Create outcome vector
y = hf_primary_model_raw["hospital_mortality"].copy()

print("Predictor matrix shape:")
print(X.shape)

print("\nOutcome vector shape:")
print(y.shape)

print("\nMissing values per predictor:")
print(X.isnull().sum())

print("\nOutcome distribution:")
print(y.value_counts())

print("\nStage 5A dataset preparation completed successfully.")

In [ ]:
# Stage 5D: Encode gender for machine learning
X["gender"] = X["gender"].map({"F": 0, "M": 1})

print("Encoded gender values:")
print(X["gender"].value_counts())

In [ ]:
# Stage 5B: Missing-value percentages

missing_summary = pd.DataFrame({
    "missing_count": X.isnull().sum(),
    "missing_percent": (
        X.isnull().sum() / len(X) * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "missing_percent",
        ascending=False
    )
)

print("\nMissing-value summary completed successfully.")

In [ ]:
# Stage 5C: Patient-level train/test split - done BEFORE imputation
# Many patients have more than one heart-failure admission.
# Splitting by patient (subject_id) keeps every admission from the same
# patient in EITHER the training set OR the test set, never both.
# This prevents the model being tested on patients it has already seen.
from sklearn.model_selection import StratifiedGroupKFold

groups = hf_primary_model_raw.loc[X.index, "subject_id"]

# 5 folds -> the first fold gives roughly an 80/20 split,
# stratified by mortality and grouped by patient
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
# Pre-imputation copies: NN validation will split these before fitting its own
# median imputer and scaler (so validation cannot inform preprocessing).
X_train_raw, X_test_raw = X_train.copy(), X_test.copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# Safety check: no patient should appear in both sets
train_patients = set(groups.iloc[train_idx])
test_patients = set(groups.iloc[test_idx])
print("Patients in both train and test:", len(train_patients & test_patients))
assert not (train_patients & test_patients), "Patient leakage across test split."

print("\nTraining predictor matrix shape:", X_train.shape)
print("Testing predictor matrix shape:", X_test.shape)
print("Training patients:", len(train_patients), "| Testing patients:", len(test_patients))

print("\nTraining mortality rate:", round(y_train.mean() * 100, 2), "%")
print("Testing mortality rate:", round(y_test.mean() * 100, 2), "%")


In [ ]:
# Stage 5D: Median imputation fit on X_train only, applied to both

# Identify numerical biomarker columns
biomarker_columns = [
    "bicarbonate",
    "chloride",
    "creatinine",
    "glucose",
    "hemoglobin",
    "platelet_count",
    "potassium",
    "sodium",
    "urea_nitrogen_bun",
    "wbc_count"
]

# Store medians for documentation
median_values = {}

# Replace missing values using medians learned from TRAINING data only
for column in biomarker_columns:

    median_value = X_train[column].median()   # learned from X_train only

    median_values[column] = median_value

    X_train[column] = X_train[column].fillna(median_value)
    X_test[column] = X_test[column].fillna(median_value)   # same value applied to X_test

print("Median values used (calculated from X_train only):")
print(pd.Series(median_values))

print("\nRemaining missing values in X_train:")
print(X_train.isnull().sum())

print("\nRemaining missing values in X_test:")
print(X_test.isnull().sum())

print("\nStage 5D median imputation completed successfully.")

In [ ]:
# Stage 5E: Recover and encode gender correctly, using the original untouched source

# Pull the ORIGINAL gender values back in, using the preserved row index
X_train["gender"] = hf_primary_model_raw.loc[X_train.index, "gender"].map({"F": 0, "M": 1})
X_test["gender"] = hf_primary_model_raw.loc[X_test.index, "gender"].map({"F": 0, "M": 1})

print("Encoded gender values (train):")
print(X_train["gender"].value_counts())

print("\nEncoded gender values (test):")
print(X_test["gender"].value_counts())

print("\nRemaining missing gender values — train:", X_train["gender"].isnull().sum())
print("Remaining missing gender values — test:", X_test["gender"].isnull().sum())

print("\nStage 5E gender recovery and encoding completed successfully.")

In [ ]:
# Stage 5F: Feature scaling — used ONLY for Logistic Regression and the Neural Network
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),   # fit on TRAIN only
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),        # same scaler applied to TEST
    columns=X_test.columns,
    index=X_test.index
)

print("Scaled training data ready:", X_train_scaled.shape)
print("Scaled testing data ready:", X_test_scaled.shape)

In [ ]:
# Stage 6A: Logistic Regression Baseline Model 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# Create logistic regression model
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"     
)

# Train model — using SCALED data
logistic_model.fit(X_train_scaled, y_train)

# Predict on testing patients
y_pred = logistic_model.predict(X_test_scaled)

# Predict mortality probabilities
y_prob = logistic_model.predict_proba(X_test_scaled)[:, 1]

# Calculate performance
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)     # NEW

print("Logistic Regression Accuracy:")
print(round(accuracy, 4))

print("\nLogistic Regression ROC-AUC:")
print(round(auc, 4))

print("\nLogistic Regression PR-AUC:")
print(round(pr_auc, 4))

print("\nConfusion Matrix (default 0.50 probability threshold; not tuned):")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nStage 6A logistic regression completed successfully.")

In [ ]:
# Stage 6B: Random Forest Mortality Prediction Model 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Build Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"     
)

# Train model — unscaled data (trees don't need scaling)
rf_model.fit(X_train, y_train)

# Generate predictions
y_pred_rf = rf_model.predict(X_test)

# Generate mortality probabilities
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate performance
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_prob_rf)
rf_pr_auc = average_precision_score(y_test, y_prob_rf)     # NEW

print("Random Forest Accuracy:")
print(round(rf_accuracy, 4))

print("\nRandom Forest ROC-AUC:")
print(round(rf_auc, 4))

print("\nRandom Forest PR-AUC:")
print(round(rf_pr_auc, 4))

print("\nConfusion Matrix (default 0.50 probability threshold; not tuned):")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nStage 6B Random Forest completed successfully.")

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
# Stage 6C: XGBoost mortality prediction model 
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Compute the class imbalance ratio from the TRAINING set only
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
pos_weight = neg_count / pos_count     # NEW: computed, not guessed

print("Negative (survived) count in training set:", neg_count)
print("Positive (died) count in training set:", pos_count)
print("scale_pos_weight used:", round(pos_weight, 2))

# Build XGBoost model
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=pos_weight     # NEW: addresses the 95/5 imbalance
)

# Train model — unscaled data (trees don't need scaling)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_prob_xgb)
xgb_pr_auc = average_precision_score(y_test, y_prob_xgb)     # NEW

print("\nXGBoost Accuracy:")
print(round(xgb_accuracy, 4))

print("\nXGBoost ROC-AUC:")
print(round(xgb_auc, 4))

print("\nXGBoost PR-AUC:")
print(round(xgb_pr_auc, 4))

print("\nConfusion Matrix (default 0.50 probability threshold; not tuned):")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nStage 6C XGBoost completed successfully.")

In [ ]:
# Stage 6D: Compare model performance (with PR-AUC added)
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "ROC_AUC": [
        auc,
        rf_auc,
        xgb_auc
    ],
    "PR_AUC": [          # NEW
        pr_auc,
        rf_pr_auc,
        xgb_pr_auc
    ]
})

model_results = model_results.sort_values(
    by="ROC_AUC",
    ascending=False
)

print("Model performance comparison:")
display(model_results)

print("\nStage 6D model comparison completed successfully.")

In [ ]:
import shap
print(shap.__version__)

In [ ]:
# STAGE 7A: Explain XGBoost (which may not be the top ROC-AUC model)
import shap

# Create explainer from trained XGBoost model
explainer = shap.TreeExplainer(xgb_model)

print("SHAP version:")
print(shap.__version__)

print("\nExplainer type:")
print(type(explainer))

print("\nStage 7A SHAP explainer created successfully.")

In [ ]:
# STAGE 7B: Calculate SHAP values
# Calculate SHAP values for test patients
shap_values = explainer.shap_values(X_test)

print("SHAP values shape:")
print(shap_values.shape)

print("\nNumber of test patients:")
print(X_test.shape[0])

print("\nNumber of predictors:")
print(X_test.shape[1])

print("\nStage 7B SHAP value calculation completed successfully.")

In [ ]:
# STAGE 7C: Global SHAP feature importance (aggregate bar plot)
from pathlib import Path
Path("figures").mkdir(exist_ok=True)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar",
    show=False
)

plt.title("Global biomarker importance for mortality prediction")

plt.tight_layout()

plt.savefig(
    str(Path("figures") / "stage_7c_shap_global_importance.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nStage 7C SHAP global importance completed successfully.")

In [ ]:
# Stage 7D: SHAP beeswarm plot
shap.summary_plot(
    shap_values,
    X_test,
    show=False
)

plt.title("SHAP Beeswarm Plot")
plt.tight_layout()
plt.show()

print("\nStage 7D SHAP beeswarm plot completed successfully.")

In [ ]:
# Stage 7E: Create ranked biomarker importance table
import pandas as pd
import numpy as np

# Calculate mean absolute SHAP values
mean_shap = np.abs(shap_values).mean(axis=0)

# Create biomarker importance table
shap_importance = pd.DataFrame({
    "Biomarker": X_test.columns,
    "SHAP_Importance": mean_shap
})

# Sort from most important to least important
shap_importance = shap_importance.sort_values(
    by="SHAP_Importance",
    ascending=False
)

# Add ranking column
shap_importance.insert(
    0,
    "Rank",
    range(1, len(shap_importance) + 1)
)

print("Top biomarkers influencing mortality prediction:")
display(shap_importance)

print("\nStage 7E biomarker ranking table completed successfully.")

In [ ]:
# Stage 7F: individual-admission example intentionally omitted.
# MIMIC-IV data are restricted. The public repository reports only aggregate
# explanations; do not publish any patient-level record or waterfall plot.
print("Individual admission explanation excluded from GitHub version.")


In [ ]:
# Stage 7G: patient-level waterfall plot intentionally omitted.
print("Global XGBoost SHAP ranking is provided in Stage 7C/7E.")


In [ ]:
import tensorflow as tf

print(tf.__version__)

print("\nTensorFlow loaded successfully.")

In [ ]:
# Stage 8A: Build Deep Learning model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

dl_model = Sequential()

dl_model.add(Dense(
    16,
    activation="relu",
    input_shape=(X_train.shape[1],)
))

dl_model.add(Dense(
    8,
    activation="relu"
))

dl_model.add(Dense(
    1,
    activation="sigmoid"
))

print(dl_model.summary())

print("\nStage 8A Deep Learning architecture created successfully.")

In [ ]:
# Stage 8B: Compile Deep Learning model
dl_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compilation completed successfully.")

In [ ]:
# Stage 8C: patient-grouped validation; no patient is in both NN train/validation.
# Critically, fit imputer and scaler AFTER splitting, only on the NN sub-train.
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

nn_groups_train = hf_primary_model_raw.loc[X_train_raw.index, "subject_id"]
nn_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=17)
nn_train_pos, nn_val_pos = next(
    nn_folds.split(X_train_raw, y_train, groups=nn_groups_train)
)
assert not (
    set(nn_groups_train.iloc[nn_train_pos])
    & set(nn_groups_train.iloc[nn_val_pos])
), "Same patient found in NN training and validation."
assert not (set(nn_groups_train) & test_patients), "Test set leakage."

nn_imputer = SimpleImputer(strategy="median")
nn_scaler = StandardScaler()

nn_X_train = nn_imputer.fit_transform(X_train_raw.iloc[nn_train_pos])
nn_X_val = nn_imputer.transform(X_train_raw.iloc[nn_val_pos])
nn_X_test = nn_imputer.transform(X_test_raw)

nn_X_train = nn_scaler.fit_transform(nn_X_train)
nn_X_val = nn_scaler.transform(nn_X_val)
X_test_nn_scaled = nn_scaler.transform(nn_X_test)

history = dl_model.fit(
    nn_X_train,
    y_train.iloc[nn_train_pos].to_numpy(),
    epochs=20,
    batch_size=32,
    validation_data=(nn_X_val, y_train.iloc[nn_val_pos].to_numpy()),
    verbose=1
)
print("NN train patients:", nn_groups_train.iloc[nn_train_pos].nunique())
print("NN validation patients:", nn_groups_train.iloc[nn_val_pos].nunique())


In [ ]:
# Stage 8D: Evaluate Deep Learning model 
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score

dl_probs = dl_model.predict(X_test_nn_scaled, verbose=0).ravel()    

dl_auc = roc_auc_score(y_test, dl_probs)
dl_pr_auc = average_precision_score(y_test, dl_probs)     
print("Deep Learning ROC-AUC:")
print(round(dl_auc, 4))

print("\nDeep Learning PR-AUC:")
print(round(dl_pr_auc, 4))
print("\nStage 8D Deep Learning evaluation completed successfully.")

In [ ]:
# Stage 8D2: Deep Learning confusion matrix and classification report
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

dl_pred = (dl_probs >= 0.5).astype(int)

print("Deep Learning Confusion Matrix (threshold = 0.50, not tuned):")
print(confusion_matrix(y_test, dl_pred))

print("\nDeep Learning Classification Report:")
print(classification_report(y_test, dl_pred))

# Plot and save the confusion matrix as a figure
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, dl_pred,
    display_labels=["Survived", "Died"],
    cmap="Blues",
    ax=ax
)
plt.title("Confusion Matrix — Deep Learning Model")
plt.tight_layout()
plt.savefig(Path("figures") / "figure_4_13_dl_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nStage 8D2 Deep Learning confusion matrix completed successfully.")

In [ ]:
# Stage 8E: Combined ROC curve for all four models
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
fpr_dl, tpr_dl, _ = roc_curve(y_test, dl_probs)

plt.figure(figsize=(8, 6))

plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_auc:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {xgb_auc:.3f})")
plt.plot(fpr_dl, tpr_dl, label=f"Deep Learning (AUC = {dl_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Four Mortality Prediction Models")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(Path("figures") / "figure_4_9_combined_roc_curve.png", dpi=300, bbox_inches="tight")

plt.show()

print("\nStage 8E combined ROC curve completed successfully.")

## Stage 9 — Reproducible held-out evaluation

This GitHub edition focuses on **in-hospital mortality prediction at the 24-hour landmark**. The exploratory Kaplan–Meier / Cox analysis from the submitted notebook has been removed rather than presenting a different cohort without the required competing-risk caveats.

**Important:** the 0.50 classification threshold is a default, not a clinically optimised cut-off. ROC-AUC and average precision are threshold-free; Brier score and reliability curves assess probability calibration. The 95% intervals below resample **patients** (all of their admissions together), and do not correct for selection/model-development uncertainty. All plots are aggregate; raw and derived patient data must remain private.


In [ ]:
# Stage 9A: collect final held-out model probabilities in the same test row order.
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, accuracy_score
)
Path("figures").mkdir(exist_ok=True)

model_probabilities = {
    "Logistic Regression": np.asarray(y_prob).ravel(),
    "Random Forest": np.asarray(y_prob_rf).ravel(),
    "XGBoost": np.asarray(y_prob_xgb).ravel(),
    "Deep Learning": np.asarray(dl_probs).ravel()
}
y_test_array = np.asarray(y_test, dtype=int)
for model_name, probs in model_probabilities.items():
    if len(probs) != len(y_test_array) or not np.isfinite(probs).all():
        raise ValueError(f"Bad probability output for {model_name}")
    if (probs < 0).any() or (probs > 1).any():
        raise ValueError(f"Predictions outside [0,1] for {model_name}")

# Explicitly label sensitivity/specificity/precision as threshold-dependent.
evaluation_rows = []
for name, probs in model_probabilities.items():
    pred = (probs >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_array, pred, labels=[0, 1]).ravel()
    evaluation_rows.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_test_array, probs),
        "PR_AUC": average_precision_score(y_test_array, probs),
        "Brier": brier_score_loss(y_test_array, probs),
        "Accuracy_at_0_5": accuracy_score(y_test_array, pred),
        "Sensitivity_at_0_5": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity_at_0_5": tn / (tn + fp) if tn + fp else np.nan,
        "Precision_at_0_5": tp / (tp + fp) if tp + fp else np.nan
    })
model_results = pd.DataFrame(evaluation_rows).sort_values(
    "ROC_AUC", ascending=False
).reset_index(drop=True)
print("Held-out test metrics (threshold-dependent columns use 0.50):")
display(model_results.round(4))
print("Test-set PR-AUC no-skill baseline:", round(y_test_array.mean(), 4))


In [ ]:
# Stage 9B: reliability curves + Brier scores. This evaluates calibration;
# it does NOT fit/recalibrate probabilities using the held-out test set.
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
for name, probs in model_probabilities.items():
    observed, predicted = calibration_curve(
        y_test_array, probs, n_bins=10, strategy="quantile"
    )
    score = brier_score_loss(y_test_array, probs)
    ax.plot(predicted, observed, marker="o", label=f"{name} (Brier={score:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.set(xlabel="Mean predicted probability", ylabel="Observed mortality rate",
       title="Calibration of 24-hour mortality models (held-out test set)")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(Path("figures") / "figure_4_14_calibration.png", dpi=300, bbox_inches="tight")
plt.show()
print("Lower Brier is better. Class-weighting can distort raw probabilities.")


In [ ]:
# Stage 9C: 95% cluster bootstrap CIs (resample unique SUBJECT_IDs).
# If a patient is sampled twice, ALL their admissions are included twice.
# This respects repeated admissions and keeps the original model fixed.
test_subject_ids = hf_primary_model_raw.loc[X_test.index, "subject_id"].to_numpy()
assert len(test_subject_ids) == len(y_test_array)
assert not (set(test_subject_ids) & train_patients)

patient_codes, unique_patient_ids = pd.factorize(test_subject_ids, sort=False)
ordered_positions = np.argsort(patient_codes, kind="stable")
patient_counts = np.bincount(patient_codes)
patient_bounds = np.r_[0, np.cumsum(patient_counts)]
patient_row_positions = [
    ordered_positions[patient_bounds[i]:patient_bounds[i + 1]]
    for i in range(len(unique_patient_ids))
]

N_BOOTSTRAPS = 1000
rng = np.random.default_rng(42)
bootstrap_scores = {
    name: {"ROC_AUC": [], "PR_AUC": []} for name in model_probabilities
}
valid_bootstraps = 0
for _ in range(N_BOOTSTRAPS):
    sampled_patients = rng.integers(
        0, len(unique_patient_ids), size=len(unique_patient_ids)
    )
    row_positions = np.concatenate([patient_row_positions[i] for i in sampled_patients])
    outcomes_b = y_test_array[row_positions]
    if np.unique(outcomes_b).size < 2:
        continue  # ROC-AUC not defined if a resample has only one class.
    valid_bootstraps += 1
    for name, probs in model_probabilities.items():
        probs_b = probs[row_positions]
        bootstrap_scores[name]["ROC_AUC"].append(roc_auc_score(outcomes_b, probs_b))
        bootstrap_scores[name]["PR_AUC"].append(average_precision_score(outcomes_b, probs_b))
if valid_bootstraps < 0.90 * N_BOOTSTRAPS:
    raise RuntimeError("Too few bootstrap resamples contained both classes.")

ci_rows = []
for name, scores in bootstrap_scores.items():
    roc_low, roc_high = np.percentile(scores["ROC_AUC"], [2.5, 97.5])
    pr_low, pr_high = np.percentile(scores["PR_AUC"], [2.5, 97.5])
    ci_rows.append({"Model": name,
                    "ROC_AUC_CI_low": roc_low, "ROC_AUC_CI_high": roc_high,
                    "PR_AUC_CI_low": pr_low, "PR_AUC_CI_high": pr_high})
ci_table = pd.DataFrame(ci_rows)
final_results = model_results.merge(ci_table, on="Model", validate="one_to_one")
print(f"Valid subject-level bootstrap samples: {valid_bootstraps}/{N_BOOTSTRAPS}")
display(final_results.round(4))


In [ ]:
# Stage 9D: aggregate README-ready table. No patient-level outputs are saved.
def format_ci(point, low, high):
    return f"{point:.3f} [{low:.3f}, {high:.3f}]"

readme_results = final_results.copy()
readme_results["ROC-AUC [95% CI]"] = readme_results.apply(
    lambda r: format_ci(r["ROC_AUC"], r["ROC_AUC_CI_low"], r["ROC_AUC_CI_high"]), axis=1
)
readme_results["PR-AUC [95% CI]"] = readme_results.apply(
    lambda r: format_ci(r["PR_AUC"], r["PR_AUC_CI_low"], r["PR_AUC_CI_high"]), axis=1
)
readme_results["Brier score"] = readme_results["Brier"].map(lambda v: f"{v:.3f}")
readme_results = readme_results[
    ["Model", "ROC-AUC [95% CI]", "PR-AUC [95% CI]", "Brier score"]
]
print("Copy these values to README.md after your authorised data rerun:")
display(readme_results)
print("| Model | ROC-AUC [95% CI] | PR-AUC [95% CI] | Brier score |")
print("|---|---|---|---|")
for row in readme_results.itertuples(index=False, name=None):
    print("| " + " | ".join(map(str, row)) + " |")


In [ ]:
# Stage 10A: Select the HIGHEST OBSERVED test ROC-AUC descriptively.
# Do not use the test-set winner to make further modelling/tuning decisions:
# picking a winner on a single held-out split introduces selection optimism.
best_observed = final_results.sort_values("ROC_AUC", ascending=False).iloc[0]
print("Highest observed held-out ROC-AUC:", best_observed["Model"])
print("ROC-AUC:", round(best_observed["ROC_AUC"], 4))
print("95% patient-bootstrap CI:",
      f"[{best_observed['ROC_AUC_CI_low']:.3f}, {best_observed['ROC_AUC_CI_high']:.3f}]")
print("Brier score:", round(best_observed["Brier"], 4))
print("Top predictors for XGBoost only (not necessarily the best-scoring model):")
display(shap_importance.head(5))
print("SHAP measures model associations, not causal effects.")


In [ ]:
# Stage 10B / Stage 11: reproducible, limited project summary.
project_summary = pd.DataFrame({
    "Finding": [
        "Research task", "Eligible admissions", "Unique eligible patients",
        "Observed in-hospital mortality", "Test-set patients",
        "Highest observed test ROC-AUC model", "Model comparison",
        "Uncertainty", "Calibration", "Threshold-based metrics",
        "Clinical use"
    ],
    "Summary": [
        "Predict in-hospital death after the first 24 hours of a HF admission",
        str(len(hf_primary_model_raw)),
        str(hf_primary_model_raw["subject_id"].nunique()),
        f"{hf_primary_model_raw['hospital_mortality'].mean():.1%}",
        str(len(test_patients)),
        str(best_observed["Model"]),
        "Four algorithms compared using ROC-AUC and PR-AUC",
        "95% cluster bootstrap CIs resample patients, not admissions",
        "Brier scores and held-out reliability plot (not recalibration)",
        "Confusion matrices use 0.50, an untuned threshold",
        "Research only: retrospective ICD-based cohort, one US hospital, "
        "no external validation or clinical deployment"
    ]
})
display(project_summary)
print("Review README cohort/results numbers after rerun; do not upload raw patient data.")
